# Deep Dive 2 — Snowflake Model Registry

**Extends:** [2.5 — Bring your own model](../Domain%202.0%20-%20Gen%20AI%20Functions/2.5.ipynb) · builds on the BYOM section of [1.1 — Gen AI principles and features](../Domain%201.0%20-%20Gen%20AI%20Overview/1.1.ipynb) · pairs with [DD1 — Snowpark Container Services](DD1%20-%20Snowpark%20Container%20Services.ipynb)

## The problem this solves

A data scientist trains a classifier in a notebook, gets a good score, and pickles it. Three months later it is running in production, nobody can say which version, the person who trained it has moved teams, and the notebook that produced it has been edited twice since. Someone asks whether the model in production is the one that was evaluated. Nobody can prove it either way.

The Model Registry is where that model goes instead: a versioned, governed schema object with its metrics attached, callable from SQL, with an audit trail of who did what.

## What you will be able to do

- Log a trained Python model as a versioned Snowflake object, with metrics attached
- Call it from SQL, using the documented `MODEL(...)!method(...)` form
- Choose between warehouse serving and container serving on the constraints that actually decide it
- Deploy a version to Snowpark Container Services without writing a Dockerfile
- Promote and roll back a version without disturbing callers

## Before you start

- `setup/dataset.sql`, for `GENAI_STUDY.PUBLIC.SUPPORT_TICKETS`.
- `snowflake-ml-python` **1.25.0 or later** for the container-serving section.
- `CREATE MODEL` on the target schema, or ownership of it.
- For the SPCS half, a compute pool and the grants from [DD1](DD1%20-%20Snowpark%20Container%20Services.ipynb). Read that deep dive first if compute pools are new to you — everything here sits on top of it.

📖 **Snowflake documentation for this notebook**
- [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)
- [Batch inference in a warehouse](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/native-batch-inference-sql)
- [Model serving in Snowpark Container Services](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/real-time-inference-rest-api)
- [Bring your own model types](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/bring-your-own-model-types)
- [MODEL_SERVING_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/model_serving_usage_history)

---
## 1. The mental model

The registry is not "the place where CPU models go, while GPUs go to SPCS". It is a **versioned model store**, and the same logged model can be served on either substrate. The choice between them is about packaging, size and latency — not about the model being good or bad.

```
train anywhere              ->  a serializable Python object (sklearn, XGBoost, PyTorch,
                                HuggingFace, a custom class)
reg.log_model(...)          ->  a MODEL object in a schema, holding one or more VERSIONS
target_platforms=[...]      ->  decides where that version can be served
    WAREHOUSE                     -> callable straight from SQL, <= 15 GB, CPU
    SNOWPARK_CONTAINER_SERVICES   -> containerised, GPU-capable, HTTP endpoint
```

| Layer | Object | How you see it |
|---|---|---|
| `Registry` | scoped to a schema | `Registry(session, database_name=…, schema_name=…)` |
| `Model` | a schema-level Snowflake object | `SHOW MODELS` |
| `ModelVersion` | a version inside a model | `SHOW VERSIONS IN MODEL` |
| method | for example `predict` | `SHOW FUNCTION IN MODEL … VERSION …` |

Being a real schema object is the whole point: models appear in `SHOW`, take ordinary grants (`USAGE` for warehouse inference, `READ` for SPCS inference and metadata), and can be granted in bulk with `GRANT USAGE ON ALL MODELS IN SCHEMA` or on future models.

### Documented limits

| Limit | Value |
|---|---|
| Versions per model | 1,000 |
| Methods per version | 10 |
| Arguments per method | 500 |
| Metadata size | 100 KB |
| Config file size | 250 KB |
| Model size for **warehouse** serving | **15 GB** |

The 1,000-version ceiling sounds generous because it is meant to be: versions are immutable, so the intended workflow is to keep old ones rather than overwrite, and promote by pointing an alias at a new one.

→ [More on the Model Registry](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

---
## 2. Logging a model

Six steps. Step 3 decides everything downstream.

| # | Step | Detail |
|---|---|---|
| 1 | Train | anywhere — a Snowflake notebook, a container runtime, your laptop |
| 2 | Open the registry | `Registry(session=…, database_name=…, schema_name=…)` |
| 3 | **Choose `target_platforms`** | `['WAREHOUSE']`, `['SNOWPARK_CONTAINER_SERVICES']`, or both |
| 4 | `log_model(...)` | with `sample_input_data` so the signature can be inferred |
| 5 | Record metrics and a comment | `mv.set_metric(...)` — this is the model card |
| 6 | Promote a default | `m.default = mv`, so callers can use an alias instead of a version name |

`log_model` requires two things: a **serializable Python object** and a `model_name` that is a valid Snowflake identifier. Dependencies go in either `conda_dependencies` or `pip_requirements` — `pip_requirements` additionally needs an `artifact_repository_map` for warehouse serving.

### Why `sample_input_data` matters more than it looks

It is how Snowflake **infers the model signature**: the column names, types and order the generated function will expect. Get it wrong and nothing fails at log time — it fails later, at call time, with a type error in a query somebody else wrote. You can supply explicit `signatures=` instead, but not both. (Models logged from Snowpark ML or MLFlow already carry a signature and need neither.)

→ [More on bring-your-own-model types](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/bring-your-own-model-types)

In [ ]:
# Step 1 — train. Nothing Snowflake-specific here.
from snowflake.snowpark.context import get_active_session
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

session = get_active_session()

df = session.table('GENAI_STUDY.PUBLIC.SUPPORT_TICKETS').to_pandas()
df = df[df['LANGUAGE'] == 'en'].dropna(subset=['TICKET_TEXT', 'CATEGORY'])

X_train, X_test, y_train, y_test = train_test_split(
    df['TICKET_TEXT'], df['CATEGORY'], test_size=0.3, random_state=42)

model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=500)),
    ('clf',   LogisticRegression(max_iter=200)),
])
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))
f1 = float(f1_score(y_test, preds, average='weighted'))
print('weighted f1:', round(f1, 4))

In [ ]:
# Steps 2-4 — open the registry and log the model
from snowflake.ml.registry import Registry
import pandas as pd

reg = Registry(session=session, database_name='GENAI_STUDY', schema_name='PUBLIC')

mv = reg.log_model(
    model            = model,
    model_name       = 'TICKET_CLASSIFIER',
    version_name     = 'V1',
    comment          = 'TF-IDF + logistic regression over SUPPORT_TICKETS',
    sample_input_data= pd.DataFrame({'TICKET_TEXT': X_test.head(3).values}),
    target_platforms = ['WAREHOUSE'],          # <- the decision that governs everything after this
    conda_dependencies = ['scikit-learn==1.5.1'],
    # pip_requirements = [...],                # pip and conda are mutually exclusive per model
    # signatures = {...},                      # explicit alternative to sample_input_data — not both
)

print('logged:', mv.fully_qualified_model_name, mv.version_name)

In [ ]:
# Step 5-6 — metrics and a default version. This is your model card.
mv.set_metric('weighted_f1', f1)
mv.set_metric('train_rows', int(len(X_train)))
mv.set_metric('feature_count', 500)

m = reg.get_model('TICKET_CLASSIFIER')
m.default = mv                       # callers can now use the LAST / default alias

print(mv.show_metrics())
print('methods:', [f.name for f in mv.show_functions()])

> ### ⚠️ Common misconceptions
>
> **"Warehouse means CPU and SPCS means GPU, so the registry is really two products."**
> It is one store. A version logged with both target platforms can be served either way, from the same model object with the same version history. What differs is packaging and hardware, not lineage — and that is exactly why the registry is worth using: the audit trail survives the deployment decision.
> → [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)
>
> **"`sample_input_data` is just an example for the docs."**
> It is what Snowflake reads to infer the signature — column names, types and order. Pass a frame whose column order differs from what callers will use, and `log_model` succeeds, the version appears, and the failure surfaces weeks later as a type error inside somebody else's query. Pass `signatures=` explicitly if you want it pinned rather than inferred; you cannot pass both.
> → [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)
>
> **"Registry models are called like `db.schema.MODEL!PREDICT(col)`."**
> The documented form wraps the model in a constructor: `MODEL(<name>)!<method>(...)`, or `MODEL(<name>, <version_or_alias>)!<method>(...)`. Learn the documented one — and note that arguments can be passed positionally or by name, but not mixed in one call.
> → [Batch inference in a warehouse](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/native-batch-inference-sql)

---
## 3. Calling the model from SQL

```sql
SELECT MODEL(<model_name>)!<method>(...)                      FROM <table>;
SELECT MODEL(<model_name>, <version_or_alias>)!<method>(...)  FROM <table>;
SELECT MODEL(my_model, LAST)!predict(...)                     FROM my_table;   -- LAST = newest version
```

Discover what you can call, and with what signature:

```sql
SHOW FUNCTION IN MODEL my_model VERSION v1;
```

Arguments may be passed positionally or by name — not both in the same call.

`LAST` is convenient and it is also a small risk: a query written against `LAST` silently changes behaviour the moment someone logs a new version. Pin the version in anything you would be unhappy to see change without warning, and use an alias you control for the places where you *do* want one flip to move every caller at once.

→ [More on batch inference in a warehouse](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/native-batch-inference-sql)

In [ ]:
%%sql -r model_versions
-- What versions exist?
SHOW VERSIONS IN MODEL GENAI_STUDY.PUBLIC.TICKET_CLASSIFIER;

In [ ]:
%%sql -r model_functions
-- What can I call on this version, and with what signature?
SHOW FUNCTION IN MODEL GENAI_STUDY.PUBLIC.TICKET_CLASSIFIER VERSION V1;

In [ ]:
%%sql -r model_predict_sql
-- Batch inference in a warehouse. LAST resolves to the newest version.
SELECT
    ticket_id,
    ticket_text,
    MODEL(GENAI_STUDY.PUBLIC.TICKET_CLASSIFIER, LAST)!predict(ticket_text) AS prediction,
    category AS actual
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en'
LIMIT 10;

---
## 4. Warehouse or containers — the real decision

| | `WAREHOUSE` | `SNOWPARK_CONTAINER_SERVICES` |
|---|---|---|
| Called from | SQL, directly | an HTTP endpoint on the service |
| Hardware | CPU | CPU **or GPU** |
| Model size | **≤ 15 GB**, and lower on small warehouses | bounded by the node |
| Dependencies | Snowflake's package set | anything — Snowflake builds the image |
| Latency shape | batch-friendly | online serving |
| Cost | warehouse credits for the query that ran | compute-pool node-hours, for as long as the service exists |
| Idle cost | none | **yes** — see [DD1](DD1%20-%20Snowpark%20Container%20Services.ipynb) |

**Choose containers when** the model exceeds 15 GB, needs a GPU, needs a package outside Snowflake's set, or must answer an HTTP request in milliseconds.
**Choose the warehouse when** it is a CPU model doing batch scoring inside SQL — which is most of them, most of the time.

The line that decides it is usually the last row of that table. Warehouse serving costs you nothing between queries. A container service is a pool node running whether or not anyone calls the model, which is the right trade only if somebody is calling it often enough, or urgently enough, to justify the wait you removed.

A version logged with **both** target platforms can be served either way. `target_platforms` is set at `log_model` time, so changing it means logging the model again — which is cheap, and gives you a new version number that makes the change visible.

> ### 🤔 Stop and think
>
> - `MODEL(my_model, LAST)!predict(...)` means a new version reaches production the moment it is logged. That is either excellent deployment hygiene or an unreviewed change, depending on who can log a version. Which is it in your organisation, and what would you have to change to make the other one true?
> - Versions are immutable and you get a thousand of them. So what is your actual retention policy — and who notices the storage if the answer is "we keep everything forever"?
> - Serving from a container removes cold starts and adds a bill that runs overnight. Above what request rate does that stop being a waste? Try to answer it in requests per hour, not in adjectives.
> - The registry stores metrics you set yourself. Nothing forces those metrics to have been computed on a held-out set. What stops a model card in your team from being flattering rather than true?

---
## 5. Deploying a version to containers

You do not write a Dockerfile. That is the point of this path — Snowflake builds the image, pushes it, and creates the SPCS service for you.

```python
mv.create_service(
    service_name             = 'ticket_classifier_svc',
    service_compute_pool     = 'ML_GPU_POOL',   # or SYSTEM_COMPUTE_POOL_CPU / SYSTEM_COMPUTE_POOL_GPU
    ingress_enabled          = True,            # True -> public HTTP endpoint
    gpu_requests             = '1',             # None (default) means CPU
    cpu_requests             = '2',             # default: all the node's CPU
    memory_requests          = '8Gi',           # default: all the node's memory
    num_workers              = 4,               # default: CPU 2*cores+1, GPU 1
    image_build_compute_pool = 'ML_BUILD_POOL', # default: the service pool
)
```

Those are the required and commonly used arguments; the `ModelVersion` API reference carries the full list.

### Prerequisites, in order

1. `snowflake-ml-python` **1.25.0 or later**
2. The model version is already logged
3. A compute pool exists and your role has `USAGE` or `OWNERSHIP` on it — or use the system pools, `SYSTEM_COMPUTE_POOL_CPU` and `SYSTEM_COMPUTE_POOL_GPU`
4. `BIND SERVICE ENDPOINT` on the account
5. `OWNER` or `READ` on the model

### What happens when you call it

```
create_service()
   -> Snowflake builds a container image for your model   (on image_build_compute_pool)
   -> creates an SPCS service on service_compute_pool
   -> exposes an HTTP endpoint on port 5000, named "inference"
```

The port and the endpoint name are fixed and cannot be customised. `image_build_compute_pool` defaults to the service pool, which means the build competes with serving for the same nodes — giving the build its own small pool is the usual reason to set it.

→ [More on model serving in SPCS](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/real-time-inference-rest-api)

In [ ]:
# Deploy the logged version to a container service for online inference.
# You write no Dockerfile: Snowflake builds the image and creates the SPCS service.
mv = reg.get_model('TICKET_CLASSIFIER').version('V1')

mv.create_service(
    service_name         = 'TICKET_CLASSIFIER_SVC',
    service_compute_pool = 'SPCS_POOL',   # or SYSTEM_COMPUTE_POOL_CPU
    ingress_enabled      = True,          # public HTTP endpoint -> needs BIND SERVICE ENDPOINT
    gpu_requests         = None,          # '1' plus a GPU pool for a GPU model
    cpu_requests         = '2',
    memory_requests      = '8Gi',
    num_workers          = 4,
)

In [ ]:
# Which services serve this model version, and at what URL?
# The endpoint host in the output is what an external caller needs.
print(mv.list_services())

In [ ]:
# Inference from Python, in a warehouse. The substrate is chosen by where you run it:
# this call goes through the warehouse, not through the service created above.
import pandas as pd
sample = pd.DataFrame({'TICKET_TEXT': ['My card was charged twice for one order.']})

print(mv.run(sample, function_name='predict'))

# To exercise the container service instead, call its HTTP endpoint -- the host comes
# from mv.list_services() or SHOW ENDPOINTS IN SERVICE. See the next cell for the shape.

### Calling the deployed endpoint from outside Snowflake

```bash
curl -X POST "https://<unique-service-id>-<account-id>.snowflakecomputing.app/predict" \
  -H 'Authorization: Snowflake Token="<pat_token>"' \
  -H 'Content-Type: application/json' \
  -d '{"dataframe_split": {"columns": ["TICKET_TEXT"], "data": [["Charged twice"]]}}'
```

- The host follows the form `https://<unique-service-id>-<account-id>.snowflakecomputing.app/<method-name>`, and comes from `mv.list_services()` or `SHOW ENDPOINTS IN SERVICE`.
- Authentication is a programmatic access token in the `Authorization: Snowflake Token="…"` header.
- The body uses `dataframe_split` (recommended) or `dataframe_records`, with an optional `params` key.
- `ingress_enabled=True` is what creates this endpoint at all. Without it the service exists and has no public door.

> ### ⚠️ Common misconceptions
>
> **"`create_service` needs me to build and push a Docker image first."**
> Snowflake builds the image for you, on `image_build_compute_pool` (defaulting to the service pool) — that is the whole reason this path exists next to raw SPCS. What you still need is a compute pool, `BIND SERVICE ENDPOINT` if the endpoint is public, and `READ` or ownership on the model.
> → [Model serving in SPCS](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/real-time-inference-rest-api)
>
> **"Deleting the service deletes the model."**
> They are separate objects. Removing a service stops the serving and the node-hours; the model version, its metrics and its history stay in the registry and can still be called from a warehouse if it was logged for that platform. The reverse trap is worse: dropping the service is what stops the bill, and suspending a query does not.
> → [MODEL_SERVING_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/model_serving_usage_history)

In [ ]:
# Registry housekeeping
print(reg.show_models())

m = reg.get_model('TICKET_CLASSIFIER')
print(m.show_versions())
print('default version:', m.default.version_name)

# Remove a service serving this version (the model itself stays)
# mv.delete_service('TICKET_CLASSIFIER_SVC')

# Export the artifact, e.g. for an audit
# mv.export('/tmp/ticket_classifier_v1')

In [ ]:
%%sql -r model_serving_cost
-- Registry serving cost
SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.MODEL_SERVING_USAGE_HISTORY
ORDER BY 1 DESC
LIMIT 50;

---
## 6. The lifecycle in one picture

```
train  ->  log_model(target_platforms=...)  ->  set_metric / comment  ->  m.default = mv
                          |
             +------------+------------+
             |                         |
        WAREHOUSE                    SPCS
             |                         |
   MODEL(name, LAST)!predict()    mv.create_service(...)
      batch scoring from SQL       HTTP endpoint on port 5000
                                   (ingress_enabled=True)
```

### Promotion, and how to undo it

1. Log `V2` alongside `V1`. Versions are immutable, so nothing existing is disturbed.
2. Compare `show_metrics()` across the two versions.
3. Flip `m.default = v2`. Every caller using the default alias moves at once.
4. Roll back by flipping it back. `V1` was never deleted.

That is what the 1,000-version ceiling is for. The cost of this pattern is that "the default" is now a production-affecting setting that anyone with the right grant can change in one line, with no review step unless you build one.

### Watching the bill

`SNOWFLAKE.ACCOUNT_USAGE.MODEL_SERVING_USAGE_HISTORY` covers both substrates, and its `INVOCATION_TYPE` column tells them apart: `SPCS` rows are an hour window of container usage and carry `COMPUTE_POOL_NAME` and `SERVICE_NAME`; `WAREHOUSE` rows are a single query and carry `FUNCTION_NAME`, `QUERY_ID` and `WAREHOUSE_ID`. `WORKLOAD_TYPE` distinguishes `SERVICE`, `JOB` and `WAREHOUSE_INFERENCE`. Credits are estimates, latency runs up to 8 hours, and 365 days are retained.

→ [More on MODEL_SERVING_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/model_serving_usage_history)

---

## Check your understanding

Twelve questions on this deep dive, weighted toward design judgement. Answer before expanding.

**1.** What decides whether a logged model can be called from SQL?

<details><summary>Show answer</summary>

`target_platforms` including `'WAREHOUSE'`, set at `log_model` time. It is not a property of the model type or of the hardware — a model logged only for `SNOWPARK_CONTAINER_SERVICES` is a perfectly good model that SQL cannot reach. Changing it means logging the model again, which produces a new version and therefore leaves a visible record of the change.

→ [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

</details>

**2.** State the documented registry limits for versions, methods and arguments, and the warehouse model size ceiling.

<details><summary>Show answer</summary>

1,000 versions per model, 10 methods per version, 500 arguments per method, and **15 GB** for warehouse serving — with the note that the practical ceiling is lower on smaller warehouses. Metadata is capped at 100 KB and the config file at 250 KB. The 15 GB figure is the one that changes an architecture; the rest are the ones that appear in recall questions.

→ [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

</details>

**3.** Which view tracks model serving cost, and how do you tell warehouse spend from container spend in it?

<details><summary>Show answer</summary>

`SNOWFLAKE.ACCOUNT_USAGE.MODEL_SERVING_USAGE_HISTORY`, using the `INVOCATION_TYPE` column, which is `SPCS` or `WAREHOUSE`. An `SPCS` row is an hour window of container usage and carries `COMPUTE_POOL_NAME` and `SERVICE_NAME`; a `WAREHOUSE` row is one query and carries `QUERY_ID`, `FUNCTION_NAME` and `WAREHOUSE_ID`. Credits are estimates and the data can lag up to 8 hours, so it is a trend tool, not a live meter.

→ [MODEL_SERVING_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/model_serving_usage_history)

</details>

**4.** A model is logged with `sample_input_data=pd.DataFrame({'text': […], 'lang': […]})`, but callers pass language first. `log_model` succeeded. What happens?

<details><summary>Show answer</summary>

The signature was inferred from the sample — those column names, those types, that order — so the generated function expects `text` then `lang`. Callers passing them the other way get a type error at call time, or worse, silently wrong predictions if both columns are strings. Nothing warns you at log time, because at log time everything was consistent. Pass `signatures=` explicitly when the calling convention is fixed by something outside your notebook.

→ [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

</details>

**5.** `SELECT GENAI_STUDY.PUBLIC.TICKET_CLASSIFIER!PREDICT(ticket_text) FROM tickets;` fails. Rewrite it.

<details><summary>Show answer</summary>

```sql
SELECT MODEL(GENAI_STUDY.PUBLIC.TICKET_CLASSIFIER, LAST)!predict(ticket_text) FROM tickets;
```

The model name goes inside a `MODEL(...)` constructor, with an optional second argument for the version or alias. `SHOW FUNCTION IN MODEL … VERSION …` tells you which methods exist and what they take, which is worth running before guessing at `predict`.

→ [Batch inference in a warehouse](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/native-batch-inference-sql)

</details>

**6.** A nightly job has used `MODEL(fraud_model, LAST)!score(...)` for a year. Overnight the results change character. What is the likely cause?

<details><summary>Show answer</summary>

Somebody logged a new version. `LAST` resolves to the newest version at query time, so logging is deployment — no review, no announcement, no change to the job's SQL. This is a feature when the team wants one flip to move everyone, and a hazard when the job is load-bearing and the logger did not know it existed. Pin a version in anything you would not want silently upgraded, and reserve alias-based promotion for the cases where the flip is the deliberate act.

→ [Batch inference in a warehouse](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/native-batch-inference-sql)

</details>

**7.** `create_service` is called with `ingress_enabled=True` and fails on privileges, though the role owns the model and the compute pool. What is missing?

<details><summary>Show answer</summary>

`BIND SERVICE ENDPOINT` on the account. It is an account-level privilege, so ownership of the model and the pool does not imply it, and only a role that can make account-level grants can hand it over. Note also that `ingress_enabled` is what creates the public endpoint at all — without it the service exists and simply has no external door, which is a legitimate configuration if only Snowflake-side callers need it.

→ [Model serving in SPCS](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/real-time-inference-rest-api)

</details>

**8.** Your model is 22 GB. What are your options, and what does each cost?

<details><summary>Show answer</summary>

Warehouse serving is out: the ceiling is 15 GB, and lower on small warehouses. That leaves serving it on containers — paying compute-pool node-hours for as long as the service exists, and giving up direct `MODEL(...)!predict()` calls from SQL in favour of an HTTP endpoint — or shrinking the model, through quantisation, pruning or a smaller architecture, and paying in accuracy and in engineering time. Which is cheaper depends entirely on call volume: a model called twice a week does not justify a pool node, and a model behind a live product does.

→ [Model serving in SPCS](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/real-time-inference-rest-api)

</details>

**9.** Why give `image_build_compute_pool` a different pool from `service_compute_pool`?

<details><summary>Show answer</summary>

Because it defaults to the service pool, so the image build competes with serving for the same nodes — a rebuild can degrade the endpoint it is meant to improve. A small, separate build pool isolates that, and can be a cheaper instance family than the one serving needs. The cost is a second pool to own, grant and remember to suspend; for a model you deploy once, the default is fine.

→ [Model serving in SPCS](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/real-time-inference-rest-api)

</details>

**10.** Your team wants a review gate before any new model reaches production. The current pattern is `m.default = mv`. What do you change?

<details><summary>Show answer</summary>

Nothing about the registry — it will not gate this for you. The flip is one line and takes effect for every caller on the default alias immediately, which is precisely why it is a good rollback mechanism and a poor approval mechanism. The change has to be around it: restrict who holds the grant that lets them set the default, run promotion through a procedure or pipeline that records an approval, and have production queries pin explicit versions so that logging a model is not the same act as shipping it. Recognise the trade — you are giving up the one-line rollback in exchange for the gate.

→ [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

</details>

**11.** You need both nightly batch scoring over ten million rows and a live endpoint for a customer-facing app, from the same model. Design it.

<details><summary>Show answer</summary>

Log the version once with **both** target platforms. Run the nightly batch through `MODEL(name, <pinned version>)!predict(...)` in a warehouse, where you pay per query and nothing runs between runs; serve the app from `create_service`, where you pay node-hours and get low latency. One model object, one version, one set of metrics, two serving paths — which is the argument for the registry over deploying two copies. What you carry is the container's idle cost, so it is worth setting the service's own `AUTO_SUSPEND_SECS` if the app has quiet hours.

→ [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

</details>

**12.** *(connects to Domain 3 — governance)* An auditor asks which model produced a decision recorded six months ago, and whether that model was evaluated before it went live. What can you show, and where does the registry stop helping?

<details><summary>Show answer</summary>

You can show the model object, its immutable versions, the metrics and comment recorded against each, and — from `MODEL_SERVING_USAGE_HISTORY` — which version was invoked, by which user, at what time, within the last 365 days. Where it stops: metrics are whatever *you* chose to record, so nothing in the registry proves they were computed on held-out data, and nothing prevents a version from being promoted without review unless you built that control yourself. The registry gives you lineage and attribution; evidence of good practice is still a process you have to run.

→ [MODEL_SERVING_USAGE_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/model_serving_usage_history)

</details>